## 4.5 章节实践

通过本章的系统学习，我们掌握了 HARQ 重传、AMC MCS 跟踪和 QoS 流控三项 MAC 层链路保障机制。现提供以下综合实践练习：

**HARQ + AMC 联合仿真**，在 Rayleigh 衰落下同时实现 HARQ 重传和 MCS 自适应跟踪，补全 3 处空缺。

要求：

1. 补全跳频信道系数 h 的获取
2. 补全 LinkQualityTracker 的 CRC 记录
3. 补全 MCS 调整决策

完成后运行 `python harq_amc_practice.py`，观察 MCS 如何随 HARQ 重传结果动态调整。

In [ ]:
%%writefile harq_amc_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.mac.qos import LinkQualityTracker
from nearlink_sdr.phy.tx_pipeline import TxConfig
from nearlink_sdr.phy.mac_interface import iq_to_mac, mac_to_iq
from nearlink_sdr.sim.link_sim import _channel_impair

snr_db = 4.0
n_frames = 60
max_retries = 3

rng = np.random.default_rng(42)
tracker = LinkQualityTracker(window_size=16)
tracker._current_mcs = 5
mcs_hist = []

for i in range(n_frames):
    mcs_idx = tracker.current_mcs
    cfg = TxConfig(frame_type=2, mcs_index=mcs_idx, pid=0x123456,
                   whitening_seed=0x52, crc_seed=0x555555,
                   crc_len=24, ctrl_bits_len=28, pilot_interval=8)
    mac_payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    from nearlink_sdr.mac.frame import AsyncDataFrame
    frame = AsyncDataFrame(segment_type=0, data=mac_payload)
    mac_bytes = frame.pack()
    iq = mac_to_iq(mac_bytes, cfg)

    # ---- HARQ 重传 + Rayleigh 衰落（每帧/每次重传独立信道）----
    success = False
    for attempt in range(1, max_retries + 2):
        # ==== TODO 1: 生成独立 Rayleigh 信道加噪（1行）====
        rx_iq = ______________
        rx = iq_to_mac(rx_iq, cfg, len(mac_bytes))
        success = rx.crc_ok and rx.mac_payload == mac_payload
        if success:
            break

    # ==== TODO: 补全 AMC 闭环（2处空缺）====
    # TODO 2: 记录 HARQ 最终 CRC 结果
    ______________
    # TODO 3: 根据窗口 FER 应用 MCS 调整
    ______________

    mcs_hist.append(tracker.current_mcs)

# ---- MCS 轨迹 ----
plt.step(range(n_frames), mcs_hist, where="mid", "b-", lw=1.5)
plt.xlabel("Frame Index"); plt.ylabel("MCS Index")
plt.yticks(range(13))
plt.title(f"HARQ + AMC (SNR={snr_db:.0f} dB, retries={max_retries})")
plt.grid(True, ls="--", alpha=0.5); plt.show()

print(f"MCS range: {min(mcs_hist)} -> {max(mcs_hist)}")
print(f"Final MCS: {mcs_hist[-1]}")
print(f"Adjustments: {sum(1 for i in range(1, len(mcs_hist)) if mcs_hist[i] != mcs_hist[i-1])}")

执行以下命令进行编译并验证结果：


In [ ]:
!python harq_amc_practice.py

执行以下代码获取答案

In [ ]:
!cat answer/04.05_answer.txt